# 02 · NFZ feasible seeds, per-seed comparison and ablation
Table and figure numbers refer to manuscript draft v5, in which all tables are in Section 5 (there is no appendix). All figures are saved to `../figures/` at 600 dpi.

| Figure file | Illustrates (draft v5) | Style |
|---|---|---|
| `Table13_feasible_matrix.png` | Table 13: NFZ results on the nine feasible seeds | annotated matrix (feasible vs all seeds) |
| `Table14_perseed_dots.png` | Table 14: per-seed HRL-H vs DMPC in the NFZ environments | paired dot plot |
| `Table15_ablation_matrix.png` | Table 15: ablation, 3 variants × 7 metrics × 4 environments | annotated matrix |
| `Table15_ablation_radar.png` | Table 15 (profile of the three variants) | radar per environment |

In [ ]:
import sys
sys.path.insert(0, "..")          # common.py lives in the package root
from common import *
%matplotlib inline

## Table 13 · NFZ environments: feasible seeds vs all seeds
In seed 8 of both NFZ environments a task is generated inside a no-fly zone and cannot be reached. Columns 1–3 use the nine feasible seeds (seed 8 excluded); columns 4–5 repeat energy and episode length over all ten seeds so that the effect of seed 8 is visible.

In [ ]:
def feas(e, m, key, fn=np.mean):
    return fn([r[key] for s, r in enumerate(PS[e][m]) if s != 8])

COLS = [("Coverage (%)\nfeasible ↑", True, lambda v: f"{v:.1f}"), ("Energy (Wh)\nfeasible ↓", False, lambda v: f"{v:.2f}"),
        ("Steps\nfeasible ↓", False, lambda v: f"{v:.0f}"), ("Energy (Wh)\nall seeds ↓", False, lambda v: f"{v:.2f}"),
        ("Steps\nall seeds ↓", False, lambda v: f"{v:.0f}")]
fig, axs = plt.subplots(1, 2, figsize=(6.9, 3.2))
for k, e in enumerate(["open_nfz", "layered_nfz"]):
    V = np.array([[feas(e, m, "coverage_frac") * 100, feas(e, m, "energy_wh_per_task"), feas(e, m, "steps_taken"),
                   bm(e, m, "energy_wh_per_task"), bm(e, m, "steps")] for m in METH], float)
    S = np.column_stack([rank_score(V[:, j], COLS[j][1]) for j in range(5)])
    annotated_matrix(axs[k], V, S, [c[2] for c in COLS], [LAB[m] for m in METH], [c[0] for c in COLS], hl_row=7, fs=5.8)
    axs[k].set_title(f"({'ab'[k]}) {ENVN[e]}", fontsize=8, pad=24)
    axs[k].axvline(2.5, color="k", lw=1.0)
fig.subplots_adjust(left=0.085, right=0.995, top=0.80, bottom=0.10, wspace=0.28)
save(fig, "Table13_feasible_matrix")

## Table 14 · per-seed HRL-H vs DMPC in the NFZ environments
Each seed is one pair (red circle = HRL-H, blue diamond = DMPC). Seed 8 contains an unreachable task (shaded). In seeds 3 and 5 of the layered NFZ environment one HRL-H UAV makes no progress for 156 and 76 steps.

In [ ]:
seeds = np.arange(10)
fig, axs = plt.subplots(2, 2, figsize=(6.9, 4.6), sharex=True)
for j, e in enumerate(["open_nfz", "layered_nfz"]):
    for i, (key, lab) in enumerate([("energy_wh_per_task", "Energy per task (Wh)"), ("steps_taken", "Episode length (steps)")]):
        ax = axs[i, j]
        h = np.array([PS[e]["hrlh"][s][key] for s in seeds], float); d = np.array([PS[e]["dmpc"][s][key] for s in seeds], float)
        ax.axvspan(7.5, 8.5, color="#f3d9a4", alpha=0.55, zorder=0)
        for s in seeds: ax.plot([s, s], [h[s], d[s]], color="#999", lw=0.8, zorder=1)
        ax.scatter(seeds, d, marker="D", s=24, color=COL["dmpc"], edgecolor="k", linewidth=0.4, zorder=3, label="DMPC")
        ax.scatter(seeds, h, marker="o", s=24, color=COL["hrlh"], edgecolor="k", linewidth=0.4, zorder=4, label="HRL-H")
        ax.set_yscale("log"); ax.set_ylim(min(h.min(), d.min()) * 0.75, max(h.max(), d.max()) * 1.9)
        ax.grid(axis="y", lw=0.3, alpha=0.6, which="both"); ax.tick_params(labelsize=6.5)
        if j == 0: ax.set_ylabel(lab, fontsize=7.4)
        if i == 0: ax.set_title(f"({'ab'[j]}) {ENVN[e]}", fontsize=8.4)
        if e == "layered_nfz" and i == 0:
            for s in (3, 5): ax.annotate("stalled UAV", (s, h[s]), xytext=(s + 0.25, h[s] * 1.05), fontsize=5.8, ha="left", va="center", color=COL["hrlh"])
    axs[0, j].text(8, axs[0, j].get_ylim()[0] * 1.15, "unreachable\ntask", ha="center", va="bottom", fontsize=5.8)
    axs[1, j].set_xlabel("Seed", fontsize=7.4); axs[1, j].set_xticks(seeds)
h_, l_ = axs[0, 0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=2, fontsize=7, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.subplots_adjust(left=0.09, right=0.995, top=0.94, bottom=0.14, wspace=0.16, hspace=0.12)
save(fig, "Table14_perseed_dots")

## Table 15 · ablation matrix
Rows: HRL-H (MAPPO + executor), HRL-H-QMIX (learner swapped), HRL-H-NAV (executor removed = MAPPO). Colour = rank among the three variants.

In [ ]:
MET = [
    ("coverage", "Coverage\n(%) ↑", True, lambda v: f"{v:.0f}"),
    ("stranded", "Stranded\n(%) ↓", False, lambda v: f"{v:.0f}"),
    ("steps", "Episode\nlength ↓", False, lambda v: f"{v:.0f}"),
    ("energy_wh_per_task", "Energy\n(Wh/task) ↓", False, lambda v: f"{v:.2f}"),
    ("dist_per_task_m", "Distance\n(m/task) ↓", False, lambda v: f"{v/1000:.1f}k" if v >= 1000 else f"{v:.0f}"),
    ("soc", "Final\nSOC ↑", True, lambda v: f"{v:.2f}"),
    ("contention_pct", "Contention\n(%) ↓", False, lambda v: f"{v:.0f}"),
]
VAR = ["hrlh", "hrlh_qmix", "hrlh_nav"]
VLAB = ["HRL-H", "HRL-H-QMIX", "HRL-H-NAV\n(= MAPPO)"]
fig, axs = plt.subplots(2, 2, figsize=(6.9, 4.3))
for k, e in enumerate(ENVS):
    ax = axs[k // 2, k % 2]
    V = np.array([[bm(e, m, key, HRLH) for key, *_ in MET] for m in VAR], float)
    S = np.column_stack([rank_score(V[:, j], MET[j][2]) for j in range(len(MET))])
    annotated_matrix(ax, V, S, [f for *_, f in MET], VLAB, [l for _, l, *_ in MET], hl_row=0, fs=5.8)
    ax.set_title(f"({'abcd'[k]}) {ENVN[e]}", fontsize=8, pad=22)
fig.subplots_adjust(left=0.115, right=0.995, top=0.86, bottom=0.15, wspace=0.28, hspace=0.62)
cax = fig.add_axes([0.32, 0.065, 0.36, 0.016])
cb = fig.colorbar(plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(0, 1)), cax=cax, orientation="horizontal")
cb.set_ticks([0, 1]); cb.set_ticklabels(["worst rank", "best rank"]); cb.ax.tick_params(labelsize=6.4, length=0); cb.outline.set_visible(False)
save(fig, "Table15_ablation_matrix")

## Table 15 · ablation radar
Scores relative to the best of the three variants in each environment (1 = best).

In [ ]:
vcol = {"hrlh": COL["hrlh"], "hrlh_qmix": "#ff7f0e", "hrlh_nav": "#9467bd"}
vname = {"hrlh": "HRL-H (MAPPO + executor)", "hrlh_qmix": "HRL-H-QMIX", "hrlh_nav": "HRL-H-NAV (= MAPPO)"}
vsty = {"hrlh": (2.0, "-"), "hrlh_qmix": (1.3, "--"), "hrlh_nav": (1.3, "-")}
fig = plt.figure(figsize=(6.9, 6.2))
for k, e in enumerate(ENVS):
    ax = fig.add_subplot(2, 2, k + 1, projection="polar")
    sc = profile_scores([rec(HRLH, e, v) for v in VAR])
    ang = radar_axes_setup(ax)
    for i, v in enumerate(VAR):
        r = np.r_[sc[i], sc[i][0]]
        ax.plot(ang, r, color=vcol[v], lw=vsty[v][0], ls=vsty[v][1], label=vname[v], zorder=10 if v == "hrlh" else 5)
        if v == "hrlh": ax.fill(ang, r, color=vcol[v], alpha=0.10)
    ax.set_title(f"({'abcd'[k]}) {ENVN[e]}", fontsize=8, pad=20)
h, l = ax.get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=3, fontsize=7, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.subplots_adjust(left=0.06, right=0.94, top=0.92, bottom=0.08, wspace=0.30, hspace=0.42)
save(fig, "Table15_ablation_radar")